# ❤️ Cardiovascular Disease Prediction — Beginner-friendly ML Notebook

Ei notebook-e amra **step by step** shikhbo:

1. Data load kora ebong bujha (EDA)
2. Data **clean** kora (age days → years, obastob blood pressure/height/weight remove)
3. **Train / Validation / Test** split kora
4. 7ta algorithm train kora: Logistic Regression, Decision Tree, Linear Regression, SVM, XGBoost, KNN, CNN
5. Shob model-er result compare kora ebong `model_comparison.csv` save kora

**Dataset:** Kaggle-er *Cardiovascular Disease dataset* (`cardio_train.csv`, 70,000 patient, real medical examination data).

**Target:** `cardio` → `1` = cardiovascular disease ache, `0` = nai.

**Kivabe chalabe:** Prottek code cell-er bam pashe ▶️ (ba `Shift + Enter`). **Upor theke niche, ek ek kore, order e** chalabe.

## Step 0: Library import kora

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay)

print("Library import hoye geche ✅")

ModuleNotFoundError: No module named 'pandas'

## Step 1: Dataset load kora

**Option A (shohoj): file upload.** Cell chalaole "Choose Files" asbe, `cardio_train.csv` select koro.

**Option B (Google Drive):** file ta Drive e rekhe nicher commented code use koro.

In [ ]:
from google.colab import files
uploaded = files.upload()   # cardio_train.csv select koro

# --- Option B: Google Drive (uporer 2 line comment kore eta use koro) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/ML_Project/cardio_train.csv'

In [ ]:
DATA_PATH = 'cardio_train.csv'     # Drive use korle uporer cell-er DATA_PATH ta bosao

# ⚠️ Ei file-er separator comma na, semicolon (;)
df = pd.read_csv(DATA_PATH, sep=';')
print("Rows, Columns =", df.shape)
df.head()

## Step 2: Data bujha (EDA)

In [ ]:
df.info()

In [ ]:
print("Missing values:", df.isnull().sum().sum())
df.describe().T

👆 **Dhyan dao:** `ap_hi` (systolic BP) er min **-150**, max **16,020**; `ap_lo` max **11,000**; `height` min 55 cm; `weight` min 10 kg. Eta **obastob (data entry-r bhul)**. Age-o **din (days)** e ache. Eder clean korte hobe.

In [ ]:
print(df['cardio'].value_counts())
sns.countplot(x='cardio', data=df)
plt.title('Cardio (0 = No disease, 1 = Disease)')
plt.show()

In [ ]:
# Age ke year e convert kore dekhi
df_plot = df.copy()
df_plot['age_years'] = df_plot['age'] / 365.25
sns.histplot(data=df_plot, x='age_years', hue='cardio', bins=30, kde=True)
plt.title('Age (years) vs Cardiovascular Disease')
plt.show()

## Step 3: Data cleaning

1. `id` column drop (eta sudhu serial number)
2. `age` din → **year**
3. Obastob row remove:
   - `ap_hi` (systolic) 70–250 er baire
   - `ap_lo` (diastolic) 40–200 er baire
   - `ap_hi` > `ap_lo` hote hobe (systolic diastolic er cheye boro)
   - `height` 120–220 cm er baire, `weight` 30–200 kg er baire
4. Duplicate row remove

In [ ]:
rows_before = len(df)

df = df.drop(columns='id')
df['age'] = df['age'] / 365.25          # days -> years

df = df[
    df['ap_hi'].between(70, 250) &
    df['ap_lo'].between(40, 200) &
    (df['ap_hi'] > df['ap_lo']) &
    df['height'].between(120, 220) &
    df['weight'].between(30, 200)
]
df = df.drop_duplicates().reset_index(drop=True)

print(f"Cleaning er age: {rows_before} row")
print(f"Cleaning er por: {len(df)} row  ({rows_before - len(df)} row remove hoyeche)")
print("Target balance (cardio=1):", round(df['cardio'].mean(), 4))
df.describe().T

In [ ]:
# Correlation: kon feature target-er sathe beshi jukto?
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap (cleaned data)')
plt.show()

## Step 4: Features (X) ar Target (y)

Shob column ekhon number, tai one-hot encoding dorkar nei (`cholesterol` ar `gluc` 1-2-3 scale, ordered category).

In [ ]:
X = df.drop(columns='cardio')
y = df['cardio']
print("Features:", list(X.columns))

## Step 5: Train / Validation / Test split

| Part | Percentage | Kaj |
|---|---|---|
| **Train** | 70% | Model ei data theke **shekhe** |
| **Validation** | 15% | Model-er settings **tune** korar jonno |
| **Test** | 15% | Shesh e **final porikkha** |

`stratify=y` mane prottek part e 0 ar 1 er anupat same thakbe.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train     :", X_train.shape)
print("Validation:", X_val.shape)
print("Test      :", X_test.shape)

## Step 6: Feature Scaling

`age` (~30-65) ar `weight` (30-200) er range alada. **Logistic Regression, Linear Regression, SVM, KNN, CNN** scale-er upor sensitive, tai **StandardScaler** use kori.

⚠️ Scaler shudhu **train data te `fit`**, val/test e shudhu `transform`, nahole data leakage hoy.

> Decision Tree ar XGBoost scaling lage na, tara original `X_train` use korbe.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
print("Scaling done ✅")

## Step 7: Result track korar helper function

**Metric gulo:**
- **Accuracy:** koyta prediction thik
- **Precision:** "disease ache" bolle koyta bar sotti thik
- **Recall:** asol rogi-der moddhe koyjon-ke dhorte perechhe (medical e khub dorkari)
- **F1:** Precision ar Recall er balance
- **ROC-AUC:** 0 ar 1 ke kototuku alada korte pare (1.0 = perfect, 0.5 = random)

In [ ]:
results = []

def evaluate(name, y_train_pred, y_val_pred, y_test_pred, y_test_score):
    row = {
        'Model': name,
        'Train Acc': accuracy_score(y_train, y_train_pred),
        'Val Acc':   accuracy_score(y_val,   y_val_pred),
        'Accuracy':  accuracy_score(y_test,  y_test_pred),
        'Precision': precision_score(y_test, y_test_pred),
        'Recall':    recall_score(y_test,    y_test_pred),
        'F1':        f1_score(y_test,        y_test_pred),
        'AUC':       roc_auc_score(y_test,   y_test_score),
    }
    results.append(row)

    print(f"===== {name} =====")
    print(f"Train Acc: {row['Train Acc']:.4f} | Val Acc: {row['Val Acc']:.4f} | Test Acc: {row['Accuracy']:.4f} | AUC: {row['AUC']:.4f}")
    print(classification_report(y_test, y_test_pred, target_names=['No Disease', 'Disease']))
    ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, cmap='Blues',
                                            display_labels=['No Disease', 'Disease'])
    plt.title(f'{name} - Confusion Matrix (Test)')
    plt.show()

# 🤖 Model 1: Logistic Regression

Nam-e "Regression" hole-o eta **classification** algorithm. Input theke 0-1 er moddhe ekta probability ber kore (0.5 er beshi → "Disease").

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_s, y_train)

evaluate('Logistic Regression',
         log_reg.predict(X_train_s), log_reg.predict(X_val_s), log_reg.predict(X_test_s),
         log_reg.predict_proba(X_test_s)[:, 1])

# 🌳 Model 2: Decision Tree

"Yes/no" proshno kore kore sidhhanto ney. **Validation diye `max_depth` tune** korbo:
- Khub choto depth → **underfitting**
- Khub boro depth → **overfitting** (train data mukhosto kore)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = range(1, 21)
train_scores, val_scores = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    val_scores.append(tree.score(X_val, y_val))

plt.plot(depths, train_scores, label='Train Accuracy', marker='o')
plt.plot(depths, val_scores, label='Validation Accuracy', marker='o')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Decision Tree: max_depth tuning'); plt.legend(); plt.grid(True)
plt.show()

best_depth = list(depths)[int(np.argmax(val_scores))]
print("Validation e shob cheye bhalo max_depth =", best_depth)

In [ ]:
dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt.fit(X_train, y_train)

evaluate('Decision Tree',
         dt.predict(X_train), dt.predict(X_val), dt.predict(X_test),
         dt.predict_proba(X_test)[:, 1])

In [ ]:
imp = pd.Series(dt.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', figsize=(8, 5), color='teal')
plt.title('Decision Tree - Feature Importance')
plt.show()

# 📈 Model 3: Linear Regression

⚠️ Linear Regression asholey **number predict** korar tool, classification-er jonno na. Amra continuous output ke **0.5 threshold** diye 0/1 e convert kori. Eta shekhar jonno / baseline hishebe, real project e Logistic Regression use kora uchit.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train_s, y_train)

def lin_predict(X_):
    return (lin_reg.predict(X_) >= 0.5).astype(int)

evaluate('Linear Regression',
         lin_predict(X_train_s), lin_predict(X_val_s), lin_predict(X_test_s),
         lin_reg.predict(X_test_s))

# ✂️ Model 4: Support Vector Machine (SVM)

Duto class-er moddhe shob cheye bhalo "sima-rekha" khuje ber kore. `kernel='rbf'` diye curved boundary-o banay.

⏳ Training data **~48,000 row**, tai ei cell **2-5 minute** nite pare. Dhoirjo dhoro!

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train_s, y_train)

evaluate('SVM',
         svm.predict(X_train_s), svm.predict(X_val_s), svm.predict(X_test_s),
         svm.decision_function(X_test_s))

# 🚀 Model 5: XGBoost

Onek choto tree ekshathe jure shoktishali model banay, proti notun tree ager tree-r bhul thik kore. `early_stopping_rounds=20` mane validation result 20 round bhalo na hole training theme jabe.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=4,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)

xgb.fit(X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False)

print("Best round:", xgb.best_iteration)

In [ ]:
hist = xgb.evals_result()
plt.plot(hist['validation_0']['logloss'], label='Train loss')
plt.plot(hist['validation_1']['logloss'], label='Validation loss')
plt.xlabel('Boosting round'); plt.ylabel('Log Loss')
plt.title('XGBoost Learning Curve'); plt.legend(); plt.grid(True)
plt.show()

In [ ]:
evaluate('XGBoost',
         xgb.predict(X_train), xgb.predict(X_val), xgb.predict(X_test),
         xgb.predict_proba(X_test)[:, 1])

# 👥 Model 6: K-Nearest Neighbors (KNN)

Notun rogi-r jonno shob cheye **K jon similar rogi** khuje tader majority vote-er moto predict kore. **Validation diye K tune** korbo.

⏳ Boro data, tai ei cell **1-3 minute** nite pare.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

k_values = [3, 5, 9, 15, 21, 25, 31, 41, 51]
knn_val_scores = []

for k in k_values:
    knn_try = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn_try.fit(X_train_s, y_train)
    knn_val_scores.append(knn_try.score(X_val_s, y_val))

plt.plot(k_values, knn_val_scores, marker='o')
plt.xlabel('K'); plt.ylabel('Validation Accuracy')
plt.title('KNN: K tuning'); plt.grid(True)
plt.show()

best_k = k_values[int(np.argmax(knn_val_scores))]
print("Shob cheye bhalo K =", best_k)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn.fit(X_train_s, y_train)

evaluate('KNN',
         knn.predict(X_train_s), knn.predict(X_val_s), knn.predict(X_test_s),
         knn.predict_proba(X_test_s)[:, 1])

# 🧠 Model 7: Deep Learning (1D CNN)

**1D CNN** diye tabular data-o kora jay: proti rogi-r feature gulo ke ekta "line" dhore `Conv1D` filter slide kore pattern khuje. `EarlyStopping` validation loss-e unnoti na hole training theme dey.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

X_train_cnn = X_train_s.reshape(-1, X_train_s.shape[1], 1)
X_val_cnn   = X_val_s.reshape(-1, X_val_s.shape[1], 1)
X_test_cnn  = X_test_s.reshape(-1, X_test_s.shape[1], 1)

n_features = X_train_s.shape[1]

cnn = models.Sequential([
    layers.Input(shape=(n_features, 1)),
    layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                     restore_best_weights=True)

history = cnn.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='Train')
ax[0].plot(history.history['val_accuracy'], label='Validation')
ax[0].set_title('CNN Accuracy'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(history.history['loss'], label='Train')
ax[1].plot(history.history['val_loss'], label='Validation')
ax[1].set_title('CNN Loss'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.show()

In [ ]:
def cnn_predict(X_):
    prob = cnn.predict(X_, verbose=0).ravel()
    return (prob >= 0.5).astype(int), prob

tr_pred, _        = cnn_predict(X_train_cnn)
val_pred, _       = cnn_predict(X_val_cnn)
test_pred, test_p = cnn_predict(X_test_cnn)

evaluate('CNN (1D)', tr_pred, val_pred, test_pred, test_p)

# 🏆 Final Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('AUC', ascending=False).reset_index(drop=True)

# CSV: full precision (round shudhu display-te)
results_df[['Model', 'AUC', 'Accuracy', 'Precision', 'Recall', 'F1']].to_csv('model_comparison.csv', index=False)

results_df.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

results_df.set_index('Model')[['Train Acc', 'Val Acc', 'Accuracy']].plot(kind='bar', ax=ax[0])
ax[0].set_title('Train vs Validation vs Test Accuracy')
ax[0].set_ylabel('Accuracy'); ax[0].set_ylim(0.6, 0.85)
ax[0].tick_params(axis='x', rotation=30); ax[0].grid(axis='y')

results_df.set_index('Model')['AUC'].plot(kind='bar', ax=ax[1], color='teal')
ax[1].set_title('ROC-AUC (Test)')
ax[1].set_ylim(0.6, 0.85)
ax[1].tick_params(axis='x', rotation=30); ax[1].grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Colab theke CSV download korte chaile:
from google.colab import files
files.download('model_comparison.csv')

## 📝 Result kivabe porbe?

- Real medical data, tai score **~72-74%** er kachakachi hoy, ekdom 100% hoy na. Shob model prai ekoi score pay, karon feature matro 11ta (dataset-er limit).
- **Train Acc >> Test Acc** hole → overfitting (Decision Tree-te depth beshi dile eta dekha jay).
- **Recall** medical e khub dorkari: recall ~66-69% mane 100 jon rogi-r moddhe ~31-34 jon-ke model miss kore.
- **Gurutwopurno:** eta **learning project**. Real diagnosis e use kora jabe na.

## 🎯 Next steps
1. `GridSearchCV` / cross-validation (`cross_val_score`)
2. BMI feature banano (`weight / (height/100)**2`)
3. Model save kora (`joblib.dump(model, 'model.pkl')`)
4. Threshold komiye recall baranor chesta